In [ ]:
import functools
import jax
from jax import numpy as jnp
from jax.sharding import PartitionSpec as P, NamedSharding, AxisType, dMesh
import optax
import orbax
import flax
from flax import nnx
import queue
import threading    
import numpy as np
from pathlib import Path

# model

In [2]:
class EncoderBlock(nnx.Module):
    def __init__(self, emb_dim, num_heads, num_kv_heads,max_seq_len,rngs:nnx.Rngs):
        self.rope=nnx.RoPE(max_seq_len=max_seq_len, embedding_size=emb_dim//num_heads)
        self.attention=nnx.MultiHeadAttention(
            in_features=emb_dim,
            num_heads=num_heads, 
            num_kv_heads=num_kv_heads,
            decode=False,
            attention_fn=functools.partial(
                nnx.dot_product_attention_with_rope, rope=self.rope
            ),
            rngs=rngs
        )
        self.l1=nnx.Linear(emb_dim,emb_dim*4,rngs=rngs)
        self.l2=nnx.Linear(emb_dim*4,emb_dim,rngs=rngs)
        self.l3=nnx.Linear(emb_dim,emb_dim,rngs=rngs)
        self.norm1=nnx.LayerNorm(emb_dim,rngs=rngs)
        self.norm2=nnx.LayerNorm(emb_dim,rngs=rngs)

    def __call__(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask=mask)
        x = self.norm2(x)
        x = nnx.swish(self.l1(x))
        x = nnx.swish(self.l2(x))
        x = x + self.l3(x)
        return x

In [3]:
class DecoderBlock(nnx.Module):
    def __init__(self, emb_dim, num_heads, num_kv_heads, max_seq_len, rngs: nnx.Rngs):
        
        self.norm1 = nnx.LayerNorm(emb_dim, rngs=rngs)
        self.rope = nnx.RoPE(max_seq_len=max_seq_len, embedding_size=emb_dim//num_heads)
        self.attention = nnx.MultiHeadAttention(
            in_features=emb_dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            attention_fn=functools.partial(
                nnx.dot_product_attention_with_rope, rope=self.rope
            ),
            decode=False,
            rngs=rngs,
        )


        self.norm_cross = nnx.LayerNorm(emb_dim, rngs=rngs)
        self.cross_attn = nnx.MultiHeadAttention(
            in_features=emb_dim,
            num_heads=num_heads,
            decode=False,
            rngs=rngs,
        )

        self.norm2 = nnx.LayerNorm(emb_dim, rngs=rngs)
        self.l1 = nnx.Linear(emb_dim, emb_dim * 4, rngs=rngs)
        self.l2 = nnx.Linear(emb_dim * 4, emb_dim * 4, rngs=rngs)
        self.l3 = nnx.Linear(emb_dim * 4, emb_dim, rngs=rngs)

    def __call__(self, x, enc_out, self_mask=None, enc_mask=None):

        x = x + self.attention(self.norm1(x), mask=self_mask, is_causal=True)
        x = x + self.cross_attn(self.norm_cross(x), enc_out, mask=enc_mask)

        h = self.norm2(x)
        h = nnx.swish(self.l1(h))
        h = nnx.swish(self.l2(h))
        x = x + self.l3(h)
        return x

In [4]:
class Transformer(nnx.Module):
    def __init__(self, vocab_size, emb_dim, num_heads, num_kv_heads, num_layers, max_seq_len, rngs: nnx.Rngs):
        # audio side
        self.conv1 = nnx.Conv(1, emb_dim, kernel_size=(3,), padding="SAME", rngs=rngs)
        self.conv2 = nnx.Conv(emb_dim, emb_dim, kernel_size=(3,), strides=(2,),
                              padding="SAME", rngs=rngs)
        self.encoder_blocks = nnx.List([
            EncoderBlock(emb_dim, num_heads, num_kv_heads, max_seq_len, rngs=rngs)
            for _ in range(num_layers)
        ])
        self.enc_ln = nnx.LayerNorm(emb_dim, rngs=rngs)      # was missing

        # text side
        self.emb = nnx.Embed(vocab_size, emb_dim, rngs=rngs)
        self.decoder_blocks = nnx.List([
            DecoderBlock(emb_dim, num_heads, num_kv_heads, max_seq_len, rngs=rngs)
            for _ in range(num_layers)
        ])
        self.ln_f = nnx.LayerNorm(emb_dim, rngs=rngs)
        self.head = nnx.Linear(emb_dim, vocab_size, rngs=rngs)

    def encode(self, audio, enc_valid=None):
        x = nnx.gelu(self.conv1(audio))
        x = nnx.gelu(self.conv2(x))
        mask = None
        if enc_valid is not None:
            mask = nnx.make_attention_mask(enc_valid, enc_valid)
        for block in self.encoder_blocks:
            x = block(x, mask=mask)
        return self.enc_ln(x)

    def decode(self, tokens, enc_out, enc_valid=None):
        x = self.emb(tokens)                                 
        cross_mask = None
        if enc_valid is not None:
            q_valid = jnp.ones(tokens.shape, dtype=bool)
            cross_mask = nnx.make_attention_mask(q_valid, enc_valid)
        for block in self.decoder_blocks:
            x = block(x, enc_out, self_mask=None, enc_mask=cross_mask)
        return self.head(self.ln_f(x))

    def __call__(self, text, audio, audio_valid=None):
        enc_valid = None if audio_valid is None else audio_valid[:, ::2]
        enc_out = self.encode(audio, enc_valid)
        return self.decode(text, enc_out, enc_valid)

# training

In [5]:
rngs = nnx.Rngs(jax.random.PRNGKey(0))
model=Transformer(vocab_size=1024, emb_dim=512, num_heads=8, num_kv_heads=4, num_layers=8, max_seq_len=1024, rngs=rngs)
tx = optax.contrib.muon(learning_rate=0.02)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)

In [6]:
@nnx.jit
def train_step(model, optimizer, text_in, audio_in, text_targets,
               loss_mask=None, audio_valid=None):
    def loss_fn(m):
        logits = m(text_in, audio_in, audio_valid=audio_valid)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, text_targets)
        if loss_mask is not None:                       # [B, T_text], 1 = real token
            loss = (loss * loss_mask).sum() / (loss_mask.sum() + 1e-8)
        else:
            loss = loss.mean()
        return loss, logits

    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model, grads)
    return loss, logits

In [7]:
param_count = sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(model, nnx.Param)))
print(f"Total parameters: {param_count:,}")

state = nnx.state(model, nnx.Param)
state = jax.tree_util.tree_map(
    lambda x: x.astype(jnp.bfloat16) if x.dtype == jnp.float32 else x, state
)
nnx.update(model, state)

Total parameters: 92,161,536


# data

In [ ]:
ROOT = "/kaggle/input/YOUR_DATASET/librispeech_460"
SR = 16000
AUDIO_STEP = 4 * SR 
TEXT_STEP = 64            
BOS, EOS, PAD = 1024, 1025, 0
 
 
class Data:
    def __init__(self, root=ROOT):
        dirs = sorted(Path(root).glob("shard_*"))
        assert dirs, f"no shard_* folders in {root}"
        self.audio = [np.load(d / "audio.npy", mmap_mode="r") for d in dirs]
        self.tokens = [np.load(d / "tokens.npy") for d in dirs]
        self.a_off = [np.r_[0, np.cumsum(np.load(d / "lens.npy"))] for d in dirs]
        self.t_off = [np.r_[0, np.cumsum(np.load(d / "tok_lens.npy"))] for d in dirs]
        # one entry per utterance: (shard, index inside shard)
        self.items = [(k, i) for k in range(len(dirs)) for i in range(len(self.a_off[k]) - 1)]
        self.dur = np.concatenate([np.diff(o) for o in self.a_off])           # samples per utterance
 
    def __len__(self):
        return len(self.items)
 
    def make(self, idx):
        """Build one padded batch from utterance numbers `idx`."""
        B = len(idx)
        T = -(-int(self.dur[idx].max()) // AUDIO_STEP) * AUDIO_STEP
        wav = np.zeros((B, T), np.int16)
        toks = []
        for r, j in enumerate(idx):
            k, i = self.items[j]
            a = self.audio[k][self.a_off[k][i]:self.a_off[k][i + 1]]
            wav[r, :len(a)] = a
            toks.append(self.tokens[k][self.t_off[k][i]:self.t_off[k][i + 1]])
 
        L = -(-(max(map(len, toks)) + 1) // TEXT_STEP) * TEXT_STEP   # +1 for BOS/EOS
        text_in = np.full((B, L), PAD, np.int32)
        targets = np.full((B, L), PAD, np.int32)
        mask = np.zeros((B, L), np.float32)
        for r, t in enumerate(toks):
            n = len(t)
            text_in[r, 0] = BOS
            text_in[r, 1:n + 1] = t
            targets[r, :n] = t
            targets[r, n] = EOS
            mask[r, :n + 1] = 1.0
        valid = np.arange(T)[None, :] < self.dur[idx][:, None]
        return wav, valid, text_in, targets, mask
 
    def epoch(self, batch_size, seed, sharding=None, prefetch=3):
        """Yield batches for one epoch. Batches are built in a background thread."""
        if sharding is not None:
            assert batch_size % sharding.mesh.size == 0, \
                f"batch_size must be divisible by {sharding.mesh.size} devices"
        rng = np.random.default_rng(seed)
        # sort by length (with jitter so batches differ each epoch), cut into batches, shuffle batches
        order = np.argsort(self.dur + rng.uniform(0, 2 * SR, len(self.dur)))
        chunks = [order[s:s + batch_size] for s in range(0, len(order) - batch_size + 1, batch_size)]
        rng.shuffle(chunks)
 
        q = queue.Queue(prefetch)
 
        def work():
            try:
                for c in chunks:
                    b = self.make(c)
                    q.put(jax.device_put(b, sharding) if sharding is not None else b)
                q.put(None)
            except Exception as e:      # hand the error to the training loop
                q.put(e)
 
        threading.Thread(target=work, daemon=True).start()
        while (b := q.get()) is not None:
            if isinstance(b, Exception):
                raise b
            yield b
 
 
def data_sharding():
    """Split the batch axis across all devices (8 on a TPU v5e-8)."""
    mesh = Mesh(np.array(jax.devices()), ("data",))
    return NamedSharding(mesh, P("data"))
 
 
def normalize_audio(wav, valid):
    """int16 [B,T] -> float32 [B,T,1], zero mean / unit variance per utterance, padding stays 0.
    Call this INSIDE the jitted train step so the CPU does no float work."""
    x = wav.astype(jnp.float32)[..., None]
    m = valid[..., None].astype(jnp.float32)
    n = jnp.maximum(m.sum(1, keepdims=True), 1.0)
    mean = (x * m).sum(1, keepdims=True) / n
    var = (((x - mean) * m) ** 2).sum(1, keepdims=True) / n
    return (x - mean) * jax.lax.rsqrt(var + 1e-5) * m


In [10]:
# 3. Example training loop iteration
batch_text_in      = jax.random.randint(rngs.params(), (2, 32), 0, 1024)
batch_text_targets = jax.random.randint(rngs.params(), (2, 32), 0, 1024)
batch_audio = jax.random.normal(rngs.params(), (2, 100, 1))


loss, logits = train_step(model, optimizer, batch_text_in, batch_audio, batch_text_targets)
print(f"Loss: {loss}")

Loss: 7.568720817565918
